<a href="https://colab.research.google.com/github/zohaibhaider017/flyrank-ml-internship-week01/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zohaibhaider017/flyrank-ml-internship-week01/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Task type: Classification with ranking-style evaluation.

My lane is Refresh / Content Opportunity Scoring. Concretely, this is a binary classification problem — predicting whether a page is "declining" (trend_direction == "down") based on observable signals — but the output isn't consumed as a simple yes/no label. It's consumed as a ranked list, since a reviewer can only act on a limited number of pages per sprint. So the classifier's predicted probability becomes a ranking score, and the real evaluation is a ranking metric (Precision@K), not plain accuracy.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Target (proxy): is_declining_label = (trend_direction == "down")

This is a proxy, not a true future outcome — it's a bucket calculated from the current data window, not something measured after a defined decision point. A stronger version for later weeks would be: features from a prior 90-day window -> decline observed over the next 30 days, which avoids using same-window information to predict itself. For this framing exercise I'm using the starter label as-is, since it's what the pipeline already computes, but I'm flagging this proxy/leakage limitation explicitly.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Success metric: Precision@50 (with Precision@20 as a secondary check).

This matches how the output is actually used: a reviewer works through a fixed number of top-ranked pages per cycle, so what matters is "of the top 50 pages we flag, how many are actually declining" — not overall accuracy across all 30,000 pages, most of which are irrelevant to the actual decision. The starter pipeline already reports this: hand-rule Precision@50 = 0.240, random forest Precision@50 = 0.740. That comparison is the benchmark I'm building against.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


One row = one page (content_id), for one client, summarized over a 90-day observation window. This is the grain the whole lane operates on — every score, rank, and reason code applies at the page level.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zohaibhaider017/flyrank-ml-internship-week01"
REPO_DIR = "flyrank-ml-internship-week01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one content page (content_id), for one client,
# observed over a fixed 90-day window.
print(f"Rows: {len(df)}, one row = one page (content_id) at a point in time")
print(f"Unique content_id: {df['content_id'].nunique()}")
print(f"Unique client_id: {df['client_id'].nunique()}")

cols_of_interest = ["content_id", "client_id", "impressions_90d", "sessions_90d",
                     "days_since_last_update", "content_age_days", "avg_position",
                     "ctr", "trend_direction"]
df[cols_of_interest].head(5)

Rows: 30000, one row = one page (content_id) at a point in time
Unique content_id: 30000
Unique client_id: 32


,content_id,client_id,impressions_90d,sessions_90d,days_since_last_update,content_age_days,avg_position,ctr,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,17,20,187,10.6,0.76,down
1,content_a1fb4e703a9e,client_4e07408562,15320,9,25,445,20.3,0.05,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,20,141,36.5,0.09,down
3,content_331d6c4de07b,client_19581e27de,11751,78,22,463,6.2,0.49,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,14,263,44.0,0.13,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*
The starter pipeline's own baseline is a fixed rule: stale (days_since_last_update >= 180) AND visible (impressions_90d >= 500). That rule is easy to explain but crude — it treats "old and popular" as the only signal worth combining, ignoring position, CTR, engagement, word count, and content type entirely.

The random forest, trained on the same observable signals, found a Precision@50 of 0.740 versus the rule's 0.240 — about 37 of the top 50 picks correct versus 12. That's a 3x lift using information the rule simply doesn't look at.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.